# Lekcija 11 - Protokol konteksta modela (MCP)

The **Model Context Protocol (MCP)** je otvoreni standard koji omogućuje agentima da dinamički otkrivaju i koriste alate, resurse i izvore podataka pri izvođenju. Umjesto da se alati statički ugrađuju u agenta, MCP omogućuje agentima povezivanje s vanjskim poslužiteljima koji izlažu mogućnosti na zahtjev.

U ovoj lekciji naučit ćete:
- Što je MCP i zašto je važan za sustave agenata
- Kako funkcionira klijent-poslužitelj arhitektura MCP-a
- Kako izgraditi agente koji koriste otkrivanje alata u stilu MCP-a


## Postavljanje

**Preduvjeti:**
- Azure AI Foundry projekt s raspoređenim modelom
- Pokrenite `az login` za autentifikaciju


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Što je Model Context Protocol (MCP)?

MCP definira standardni način na koji AI agenti otkrivaju i stupaju u interakciju s vanjskim alatima i izvorima podataka:

- **MCP Server**: Izlaže alate, resurse i promptove putem standardnog protokola
- **MCP Client**: Okruženje izvršavanja agenta koje se povezuje na servere i otkriva dostupne mogućnosti
- **Dynamic Discovery**: Agenti ne trebaju unaprijed zapisane alate — otkrivaju što je dostupno tijekom izvođenja

Ovo je snažno za izgradnju proširivih sustava agenata gdje se nove mogućnosti mogu dodavati bez izmjene koda agenta.


## Kako MCP radi

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (MCP klijent) se povezuje na MCP poslužitelj
2. Poslužitelj odgovara s popisom dostupnih alata i njihovih shema
3. Agent tada može pozvati bilo koji otkriveni alat tijekom svog zaključivanja
4. Rezultati se vraćaju putem istog protokola


## Simuliranje otkrivanja alata MCP-a

Budući da pravi MCP poslužitelj zahtijeva pokrenut poslužiteljski proces, prikazat ćemo obrazac koristeći `@tool` funkcije koje simuliraju ono što bi usluga smještaja povezana s MCP-om pružila.

U produkciji, ovi alati bi se otkrivali dinamički s MCP poslužitelja umjesto da su definirani lokalno.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Izgradnja agenta s alatima u MCP stilu


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP u produkciji

U produkcijskom okruženju, MCP omogućuje moćne obrasce:

- **Dinamično otkrivanje alata**: Agenti se povezuju na MCP poslužitelje i otkrivaju alate tijekom izvođenja
- **Odvojena arhitektura**: Pružatelji alata mogu ažurirati svoje komponente neovisno o agentu
- **Dijeljenje između organizacija**: Timovi mogu izložiti mogućnosti putem MCP poslužitelja koje može koristiti bilo koji agent
- **Podrška Microsoft Agent Frameworka**: MAF ima ugrađenu podršku za MCP klijenta putem `mcp` integracije

Za korištenje stvarnog MCP poslužitelja s MAF-om, povezali biste se putem `hosted_mcp_tool()` ili MCP klijentske integracije.

**Saznajte više:**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Sažetak

U ovoj lekciji naučili ste:
- **MCP** je otvoreni standard za dinamičko otkrivanje alata između agenata i pružatelja alata
- **arhitektura klijent-poslužitelj** omogućava agentima otkrivanje mogućnosti u vrijeme izvođenja
- MCP omogućuje **proširive, odvojene sustave agenata** u kojima se alati mogu dodavati bez promjena koda
- Microsoft Agent Framework pruža **ugrađenu podršku za MCP** za upotrebu u produkciji


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Odricanje odgovornosti**:
Ovaj dokument je preveden pomoću AI usluge za prijevod [Co-op Translator](https://github.com/Azure/co-op-translator). Iako nastojimo postići točnost, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati mjerodavnim izvorom. Za kritične informacije preporučuje se profesionalni prijevod koji obavi ljudski prevoditelj. Ne snosimo odgovornost za bilo kakve nesporazume ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
